In [37]:
import pandas as pd
import sklearn

In [3]:
df = pd.read_csv(r"C:\Users\Hp\OneDrive\Desktop\hiver-support-agent\data\raw\twcs\twcs.csv")

In [4]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [5]:
df.shape

(2811774, 7)

In [6]:
df.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [8]:
df.isnull().sum()

tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64

In [9]:
df['inbound'].value_counts()

inbound
True     1537843
False    1273931
Name: count, dtype: int64

In [ ]:
support_df = df[df['inbound'] == False]

brand_counts = (
	support_df['author_id']
	.value_counts()
	.reset_index()
)

brand_counts.columns = ['brand', 'support_tweets']

brand_counts.head(20)

,brand,support_tweets
0,AmazonHelp,169840
1,AppleSupport,106860
2,Uber_Support,56270
3,SpotifyCares,43265
4,Delta,42253
5,Tesco,38573
6,AmericanAir,36764
7,TMobileHelp,34317
8,comcastcares,33031
9,British_Airways,29361


In [11]:
tweet_lookup = df.set_index('tweet_id')

In [ ]:
customer_tweets = df[
	(df['inbound'] == True) &
	(df['in_response_to_tweet_id'].notna())
].copy()

In [ ]:
customer_tweets['parent_tweet_id'] = (
	customer_tweets['in_response_to_tweet_id']
	.astype('Int64')
)

customer_tweets['brand'] = customer_tweets['parent_tweet_id'].map(
	tweet_lookup['author_id']
)

In [ ]:
brand_customer_counts = (
	customer_tweets['brand']
	.value_counts()
	.reset_index()
)

brand_customer_counts.columns = [
	'brand',
	'customer_replies'
]

brand_customer_counts.head(30)

,brand,customer_replies
0,AmazonHelp,100503
1,AppleSupport,36658
2,Uber_Support,22160
3,VirginTrains,18450
4,AmericanAir,18045
5,SpotifyCares,15096
6,Delta,14470
7,GWRHelp,13127
8,ATVIAssist,13055
9,VerizonSupport,12953


In [15]:
brand_customer_counts.head(30)

,brand,customer_replies
0,AmazonHelp,100503
1,AppleSupport,36658
2,Uber_Support,22160
3,VirginTrains,18450
4,AmericanAir,18045
5,SpotifyCares,15096
6,Delta,14470
7,GWRHelp,13127
8,ATVIAssist,13055
9,VerizonSupport,12953


In [ ]:
amazon = customer_tweets[
	customer_tweets['brand'] == 'AmazonHelp'
].copy()

amazon.shape

(100503, 9)

In [17]:
amazon['response_tweet_id'].notna().sum()

np.int64(69710)

In [18]:
amazon['response_tweet_id'].notna().mean() * 100

np.float64(69.36111359859905)

In [ ]:
amazon[
	['tweet_id', 'author_id', 'text',
	 'response_tweet_id', 'in_response_to_tweet_id']
].head(20)

,tweet_id,author_id,text,response_tweet_id,in_response_to_tweet_id
182,270,115770,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。,NaN,269.0
183,271,115770,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,269.0
185,274,115770,@AmazonHelp こちらこそありがとうございました。,275,273.0
322,616,115820,@AmazonHelp 3 different people have given 3 di...,618,615.0
324,619,115820,@AmazonHelp I frankly don't have the patience ...,NaN,618.0
329,623,115824,"@AmazonHelp Okay, danke für die Info",625,622.0
333,627,115827,@AmazonHelp @115826 Yeah this is crazy we’re l...,629,626.0
344,638,115834,@AmazonHelp Hi ready for some help,637,639.0
347,641,115834,@AmazonHelp Nothing there helped me with the E...,640,642.0
351,645,115835,@AmazonHelp That page is useless - doesn’t all...,647,644.0


In [ ]:
amazon_support = df[
	(df['author_id'] == 'AmazonHelp') &
	(df['inbound'] == False)
].copy()

amazon_support.shape

(169840, 7)

In [21]:
amazon_support.shape

(169840, 7)

In [22]:
amazon_support['response_tweet_id'].notna().sum()

np.int64(85274)

In [23]:
amazon_support['response_tweet_id'].notna().mean() * 100

np.float64(50.20843146490815)

In [ ]:
amazon_pairs = amazon[['tweet_id', 'text', 'response_tweet_id']].copy()

amazon_pairs = amazon_pairs[
	amazon_pairs['response_tweet_id'].notna()
].copy()

amazon_pairs['response_id'] = amazon_pairs['response_tweet_id'].str.split(',')
amazon_pairs = amazon_pairs.explode('response_id')

amazon_pairs['response_id'] = pd.to_numeric(
	amazon_pairs['response_id'],
	errors='coerce'
).astype('Int64')

amazon_pairs['response_text'] = amazon_pairs['response_id'].map(
	tweet_lookup['text']
)

amazon_pairs['response_author'] = amazon_pairs['response_id'].map(
	tweet_lookup['author_id']
)

amazon_pairs = amazon_pairs[
	amazon_pairs['response_author'] == 'AmazonHelp'
]

amazon_pairs.shape

(70956, 6)

In [ ]:
amazon_pairs[
	['tweet_id', 'text', 'response_text']
].head(10)

,tweet_id,text,response_text
183,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
185,274,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
322,616,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
329,623,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
333,627,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM
344,638,@AmazonHelp Hi ready for some help,@115834 Were you able to reach us at the link ...
351,645,@AmazonHelp That page is useless - doesn’t all...,@115835 You can also request a call back here ...
357,651,@AmazonHelp I don't want a form to fill out th...,@115838 Is there a current order you're having...
359,654,@AmazonHelp Already started the return. UPS ge...,@115838 Provide your details here: https://t.c...
363,657,"@AmazonHelp Already handled, just venting. It ...",@115839 Let us know if there’s anything else w...


In [ ]:
amazon_pairs = amazon_pairs[
	['text', 'response_text']
].rename(columns={
	'text': 'customer_text'
})

amazon_pairs.head()

,customer_text,response_text
183,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
185,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
322,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
329,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
333,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM


In [27]:
amazon_pairs.isnull().sum()

customer_text    0
response_text    0
dtype: int64

In [28]:
amazon_pairs.duplicated().sum()

np.int64(0)

In [29]:
amazon_pairs['customer_text'].str.len().describe()

count    70956.000000
mean       107.965894
std         62.059370
min         13.000000
25%         58.000000
50%        103.000000
75%        146.000000
max        323.000000
Name: customer_text, dtype: float64

In [30]:
amazon_pairs['customer_text'].sample(20, random_state=42).tolist()

["@AmazonHelp Les autres transporteurs, @120534, DHL, TNT trouvent pourtant bien la societé ou je travail et j'ai jamais eu de soucis de livraiso",
 '@AmazonHelp Just a refund, which has landed me in the position of trying to send the TV back without the original box.',
 '@AmazonHelp Without pausing any internet activities, I’m getting about 330 down. It’s not my bandwidth. Maybe you can forward this to your dev team. https://t.co/rKJobUAmW4',
 '@AmazonHelp The estimated date is 23rd october',
 '@AmazonHelp This is what you get on the app not he best way to ship a kids toy would be best to ship it in a box as a standard to avoid ruining suprises https://t.co/fh2mrLu0F3',
 '@AmazonHelp Ya i have already checked ,but this response is not reducing my tension ,i am not satisfied by it.i want faster action.',
 '@AmazonHelp 見ようとした時に気づきました！ありがとうございます！',
 "@AmazonHelp Thank you for the sugar coated email but that doesn't help. I wasn't wrong when I said it's a ridiculous system.",
 "@AmazonHel

In [32]:
amazon_pairs['customer_text'].sample(50, random_state=123).tolist

<bound method IndexOpsMixin.tolist of 1101684    @AmazonHelp Pls refer to the order no - 403-30...
367693     @AmazonHelp Ok.. is it possible to change the ...
626807                         @AmazonHelp Sold by Amazon...
2527123    @AmazonHelp No I'm not. I previously had a mem...
2142106    @AmazonHelp Je ne parle pas de retard de colis...
1121857    @AmazonHelp Quick response &amp; some much nee...
1141771          @AmazonHelp Both have already been refunded
1292850                                  @AmazonHelp filled.
1106170    @AmazonHelp No you do not understand the conce...
1218821    @AmazonHelp After placing order new order\nTra...
821092     @AmazonHelp And the replacement, couriered by ...
475549     @AmazonHelp I tried the chat team earlier. The...
2362910                      @AmazonHelp Shared the details.
219831     @AmazonHelp After reading T&amp;C have twisted...
805226     @AmazonHelp Can u hep me with cuatomer  are no...
655948     @AmazonHelp We have no membership - 

In [ ]:
amazon_sample = amazon_pairs.sample(
	1000,
	random_state=42
).copy()

amazon_sample.head()

,customer_text,response_text
1438738,"@AmazonHelp Les autres transporteurs, @120534,...",@454720 Vous m'en voyez sincèrement navrée. L'...
337965,"@AmazonHelp Just a refund, which has landed me...",@132658 I'd suggest finding a box locally that...
2082584,@AmazonHelp Without pausing any internet activ...,@386460 Let's troubleshoot this in real time. ...
1264195,@AmazonHelp The estimated date is 23rd october,@444279 Please wait till the estimated deliver...
2381432,@AmazonHelp This is what you get on the app no...,@431264 Thanks for sharing your feedback with ...


In [ ]:
eval_set = amazon_sample.sample(
	200,
	random_state=42
).copy()

eval_set.head()

,customer_text,response_text
1282368,@AmazonHelp 公式様から直々にお礼を言われるとは！\nありがとうございます！楽しま...,@448551 公式から突然のリプライ失礼いたしました🙇\nはい！今後ともAmazonをよろ...
792498,@AmazonHelp Thank you! 😍,@329907 Of course! We're always here if you ne...
1737824,"@AmazonHelp I have done, a number of times, bu...","@54213 Sorry! That first link is incorrect, yo..."
2319583,@AmazonHelp i had the student membership and yes,@710000 Oh no! Have you received any e-mails f...
1442752,@AmazonHelp Your executive on call promised th...,"@274956 Truly sorry about that, Monica. We’d l..."


In [41]:
eval_set.to_csv("C:/Users/Hp/OneDrive/Desktop/hiver-support-agent/data/processed/amazon_eval.csv", index=False)

In [38]:

print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.7.2


In [45]:
def assign_intent(text):
	text = text.lower()
	
	if any(word in text for word in ['refund', 'refunded', 'money back']):
		return 'refund_issue'
	elif any(word in text for word in ['return', 'returns', 'returning']):
		return 'return_issue'
	elif any(word in text for word in ['delivery', 'delivered', 'shipping', 'package', 'parcel', 'courier']):
		return 'delivery_issue'
	elif any(word in text for word in ['payment', 'paid', 'charge', 'charged', 'card']):
		return 'payment_issue'
	elif any(word in text for word in ['damaged', 'broken', 'defective', 'faulty']):
		return 'product_issue'
	elif any(word in text for word in ['login', 'log in', 'sign in', 'account', 'password', 'membership']):
		return 'account_issue'
	elif any(word in text for word in ['app', 'website', 'site', 'error', 'not working']):
		return 'technical_issue'
	elif any(word in text for word in ['agent', 'executive', 'customer service', 'call']):
		return 'contact_support'
	elif any(word in text for word in ['order', 'ordered', 'ordering']):
		return 'order_issue'
	else:
		return 'general_query'

In [ ]:
# Test the function
print(assign_intent("I want a refund for my broken item"))
print(assign_intent("Where is my package?"))
print(assign_intent("I can't log into my account"))

refund_issue
delivery_issue
account_issue
